# Keel router: fine-tune Qwen2.5-1.5B-Instruct for intent + risk classification

Runs on a free Colab T4 (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all**. No uploads or secrets needed:
the notebook downloads the public Bitext dataset, builds the same train/val/test split as `finetune/build_dataset.py` (seed 2026),
evaluates the base model (zero-shot and 10-shot), fine-tunes with QLoRA (Unsloth), evaluates again and prints `results.json`.
The large-API-model row (gpt-oss-120b on Groq) is produced locally by `finetune/compare_api.py` on the identical test set (same SHA-256).

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps "trl==0.15.2" "peft" "accelerate" "bitsandbytes"
!pip install -q scikit-learn httpx datasets

In [ ]:
pathlib_src = 'import argparse\nimport json\nimport random\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nHERE = Path(__file__).parent\nOUT = HERE / "data"\nSEED = 2026\n\nLABELS = [\n    "order_status",\n    "refund_request",\n    "cancel_order",\n    "change_address",\n    "product_question",\n    "policy_question",\n    "complaint",\n    "human_request",\n    "legal_safety",\n    "other",\n]\n\nROUTE = {\n    "order_status": "orders",\n    "refund_request": "orders",\n    "cancel_order": "orders",\n    "change_address": "orders",\n    "product_question": "support",\n    "policy_question": "support",\n    "other": "support",\n    "complaint": "escalation",\n    "human_request": "escalation",\n    "legal_safety": "escalation",\n}\n\nSYSTEM = (\n    "You classify customer messages for Hearth & Field, a UK homeware shop. "\n    "Reply with exactly one label from: " + ", ".join(LABELS) + "."\n)\n\nBITEXT_MAP = {\n    "track_order": "order_status",\n    "delivery_period": "order_status",\n    "get_refund": "refund_request",\n    "track_refund": "refund_request",\n    "cancel_order": "cancel_order",\n    "change_shipping_address": "change_address",\n    "set_up_shipping_address": "change_address",\n    "check_refund_policy": "policy_question",\n    "delivery_options": "policy_question",\n    "check_payment_methods": "policy_question",\n    "check_cancellation_fee": "policy_question",\n    "complaint": "complaint",\n    "contact_human_agent": "human_request",\n    "contact_customer_service": "human_request",\n    "create_account": "other",\n    "edit_account": "other",\n    "recover_password": "other",\n    "newsletter_subscription": "other",\n    "review": "other",\n    "check_invoice": "other",\n    "get_invoice": "other",\n}\n\nFILL = {\n    "order": [f"HF-{n}" for n in range(204100, 204990, 37)],\n    "product": [\n        "oak bed frame",\n        "linen duvet cover",\n        "goose down pillow",\n        "brass table lamp",\n        "velvet sofa",\n        "wool rug",\n        "copper kettle",\n        "stoneware dinner set",\n        "blackout curtains",\n        "pocket sprung mattress",\n        "bentwood chair",\n        "arched mirror",\n        "towel bale",\n        "extendable dining table",\n    ],\n    "place": ["Leeds", "Glasgow", "Cardiff", "Belfast", "Inverness", "Jersey", "the Isle of Wight", "Truro"],\n    "addr": [\n        "14 Mill Lane, York YO1 7HB",\n        "3 Station Road, Bath BA1 1SU",\n        "22 Queen Street, Cardiff CF10 2BU",\n        "9 Castle Wynd, Inverness IV2 3EB",\n    ],\n    "issue": ["broken", "damaged", "faulty", "scratched", "the wrong colour", "missing a leg"],\n    "harm": [\n        "caught fire",\n        "gave my son an electric shock",\n        "collapsed and injured my mum",\n        "started smoking",\n        "burned my hand",\n    ],\n    "topic": [\n        "returns",\n        "refunds",\n        "delivery",\n        "Klarna",\n        "gift cards",\n        "the price promise",\n        "warranty",\n        "student discount",\n        "NHS discount",\n        "trade accounts",\n        "home assembly",\n        "Hearth Rewards points",\n    ],\n}\n\nTRAIN_TEMPLATES = {\n    "order_status": [\n        "where is my order {order}",\n        "Has {order} been dispatched yet?",\n        "Any update on {order}? It\'s been a week",\n        "tracking for {order} hasn\'t moved since Monday",\n        "when will {order} arrive?",\n        "is my {product} on its way? order {order}",\n    ],\n    "refund_request": [\n        "I\'d like a refund for {order}, the {product} arrived {issue}",\n        "can I get my money back on {order}",\n        "please refund {order}",\n        "{order} came {issue}, refund please",\n        "has my refund for {order} gone through?",\n    ],\n    "cancel_order": [\n        "please cancel {order}",\n        "I need to cancel my order {order}",\n        "cancel {order} I ordered by mistake",\n        "can you stop {order} before it ships?",\n    ],\n    "change_address": [\n        "change the delivery address on {order} to {addr}",\n        "I\'ve moved, can {order} go to {addr}?",\n        "wrong postcode on {order}, should be {addr}",\n    ],\n    "product_question": [\n        "is the {product} in stock?",\n        "how much is the {product}?",\n        "what size is the {product}?",\n        "is the {product} machine washable?",\n        "does the {product} come assembled?",\n        "what colours does the {product} come in?",\n    ],\n    "policy_question": [\n        "how do {topic} work?",\n        "what\'s your policy on {topic}?",\n        "do you deliver to {place}?",\n        "how long do I have to return something?",\n        "can I pay with Klarna?",\n        "how much is next day delivery?",\n        "do you offer {topic}?",\n    ],\n    "complaint": [\n        "this is the third time my order is late, absolutely fed up",\n        "your service is a joke",\n        "worst shopping experience I\'ve ever had",\n        "I\'m so angry, nobody replies to my emails",\n        "disgusting customer service, sort it out",\n    ],\n    "human_request": [\n        "can I speak to a real person",\n        "put me through to a human please",\n        "I want to talk to a manager",\n        "stop the bot, get me an agent",\n    ],\n    "legal_safety": [\n        "my solicitor will be in touch about {order}",\n        "I\'m filing a chargeback with my bank",\n        "the {product} {harm}",\n        "I\'m reporting you to Trading Standards",\n        "someone used my card to order {order}, this is fraud",\n    ],\n    "other": [\n        "what\'s the weather like in {place}",\n        "who is your CEO?",\n        "do you sell garden sheds?",\n        "can you recommend a plumber",\n        "tell me a joke",\n        "how do I reset my password?",\n    ],\n}\n\nTEST_TEMPLATES = {\n    "order_status": [\n        "{order} still not here, where\'s it got to?",\n        "could you check the status of {order} for me",\n        "my parcel {order} says out for delivery since yesterday",\n    ],\n    "refund_request": [\n        "the {product} is {issue}, I want a refund on {order}",\n        "money back for {order} please",\n        "returned {order} two weeks ago, where\'s my refund?",\n    ],\n    "cancel_order": ["changed my mind, please cancel {order}", "don\'t send {order}, cancel it"],\n    "change_address": ["send {order} to {addr} instead", "update the address for {order} to {addr} please"],\n    "product_question": [\n        "have you got the {product} in grey?",\n        "what are the dimensions of the {product}?",\n        "is the {product} back in stock soon?",\n    ],\n    "policy_question": [\n        "can I return opened pillows?",\n        "is delivery free over a certain amount?",\n        "how long is the warranty on {topic}?",\n        "do you ship to {place}?",\n    ],\n    "complaint": [\n        "honestly appalling, I\'ve waited 3 weeks",\n        "fuming, the courier threw my parcel over the gate",\n        "you lot are useless",\n    ],\n    "human_request": ["I\'d rather speak to someone real", "is there a person I can talk to?"],\n    "legal_safety": [\n        "the {product} {harm}, I need this dealt with now",\n        "I\'ll be contacting the ombudsman",\n        "my lawyer says you\'re breaching the Consumer Rights Act",\n    ],\n    "other": ["what time does the football start", "can you write me a poem", "where can I buy paint?"],\n}\n\n\ndef fill(t: str, rnd: random.Random) -> str:\n    return re.sub(r"\\{(\\w+)\\}", lambda m: rnd.choice(FILL[m.group(1)]), t)\n\n\ndef clean_bitext(text: str, rnd: random.Random) -> str:\n    text = re.sub(r"\\{\\{Order Number\\}\\}", lambda _: rnd.choice(FILL["order"]), text)\n    text = re.sub(r"\\{\\{[^}]+\\}\\}", "", text)\n    return re.sub(r"\\s+", " ", text).strip()\n\n\ndef norm(t: str) -> str:\n    return re.sub(r"[^a-z]+", " ", t.lower()).strip()\n\n\ndef row(text: str, label: str, source: str) -> dict:\n    return {\n        "text": text,\n        "label": label,\n        "route": ROUTE[label],\n        "source": source,\n        "messages": [\n            {"role": "system", "content": SYSTEM},\n            {"role": "user", "content": text},\n            {"role": "assistant", "content": label},\n        ],\n    }\n\n\ndef synth(templates: dict, per_label: int, rnd: random.Random, source: str) -> list[dict]:\n    rows = []\n    for label, tpls in templates.items():\n        seen = set()\n        tries = 0\n        while len([r for r in rows if r["label"] == label]) < per_label and tries < per_label * 20:\n            tries += 1\n            t = fill(rnd.choice(tpls), rnd)\n            if rnd.random() < 0.25:\n                t = t.lower()\n            if rnd.random() < 0.15:\n                t = t.rstrip("?.!") + rnd.choice(["!!", " pls", " thanks", " asap"])\n            if norm(t) in seen:\n                continue\n            seen.add(norm(t))\n            rows.append(row(t, label, source))\n    return rows\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--bitext", required=True, help="Bitext dataset exported as JSON lines")\n    ap.add_argument("--per-label", type=int, default=220)\n    a = ap.parse_args()\n    rnd = random.Random(SEED)\n\n    by_label: dict[str, list[str]] = {lbl: [] for lbl in LABELS}\n    for line in Path(a.bitext).read_text().splitlines():\n        r = json.loads(line)\n        lbl = BITEXT_MAP.get(r["intent"])\n        if lbl:\n            by_label[lbl].append(clean_bitext(r["instruction"], rnd))\n\n    bitext_train, bitext_test = [], []\n    for lbl, texts in by_label.items():\n        uniq = list({norm(t): t for t in texts if len(t) > 6}.values())\n        rnd.shuffle(uniq)\n        bitext_test += [row(t, lbl, "bitext") for t in uniq[:10]]\n        bitext_train += [row(t, lbl, "bitext") for t in uniq[10 : 10 + a.per_label]]\n\n    synth_train = synth(TRAIN_TEMPLATES, 120, rnd, "synthetic")\n    synth_test = synth(TEST_TEMPLATES, 20, rnd, "synthetic-heldout")\n\n    train_all = bitext_train + synth_train\n    test = bitext_test + synth_test\n    test_keys = {norm(r["text"]) for r in test}\n    train_all = [r for r in train_all if norm(r["text"]) not in test_keys]\n    rnd.shuffle(train_all)\n    n_val = len(train_all) // 10\n    val, train = train_all[:n_val], train_all[n_val:]\n\n    OUT.mkdir(exist_ok=True)\n    for name, rows in (("train", train), ("val", val), ("test", test)):\n        with open(OUT / f"{name}.jsonl", "w") as f:\n            f.writelines(json.dumps(r) + "\\n" for r in rows)\n\n    stats = {\n        name: dict(Counter(r["label"] for r in rows)) for name, rows in (("train", train), ("val", val), ("test", test))\n    }\n    stats["sizes"] = {"train": len(train), "val": len(val), "test": len(test)}\n    (OUT / "stats.json").write_text(json.dumps(stats, indent=2))\n    print(json.dumps(stats["sizes"]), json.dumps(stats["test"]))\n\n\nif __name__ == "__main__":\n    main()\n'
open('build_dataset.py','w').write(pathlib_src)
from datasets import load_dataset
load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train').to_json('bitext.jsonl')
!python build_dataset.py --bitext bitext.jsonl

In [ ]:
import os, json, time, statistics, pathlib, hashlib
BASE = "unsloth/Qwen2.5-1.5B-Instruct"
DATA = pathlib.Path("data")
load = lambda n: [json.loads(l) for l in open(DATA / f"{n}.jsonl")]
train, val, test = load("train"), load("val"), load("test")
LABELS = ["order_status","refund_request","cancel_order","change_address","product_question","policy_question","complaint","human_request","legal_safety","other"]
TEST_SHA = hashlib.sha256((DATA / "test.jsonl").read_bytes()).hexdigest()
print(len(train), len(val), len(test), "test sha256", TEST_SHA)

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=512, load_in_4bit=True)

def parse(out):
    o = out.strip().lower().replace(" ", "_")
    for l in sorted(LABELS, key=len, reverse=True):
        if l in o:
            return l
    return "other"

def predict(m, rows, few_shot=False):
    FastLanguageModel.for_inference(m)
    shots = []
    if few_shot:
        seen = set()
        for r in train:
            if r["label"] not in seen and r["source"] == "synthetic":
                seen.add(r["label"]); shots += [r["messages"][1], r["messages"][2]]
    preds, lats = [], []
    for r in rows:
        msgs = [r["messages"][0]] + shots + [r["messages"][1]]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
        torch.cuda.synchronize(); t0 = time.time()
        out = m.generate(input_ids=ids, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize(); lats.append((time.time() - t0) * 1000)
        preds.append(parse(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)))
    return preds, lats

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
gold = [r["label"] for r in test]
ROUTE = {"order_status":"orders","refund_request":"orders","cancel_order":"orders","change_address":"orders","product_question":"support","policy_question":"support","other":"support","complaint":"escalation","human_request":"escalation","legal_safety":"escalation"}
GPU_USD_PER_HOUR = 0.35
def summarise(name, preds, lats, usd_per_1k=None):
    r = {"model": name, "accuracy": round(100*accuracy_score(gold, preds), 1),
         "macro_f1": round(100*f1_score(gold, preds, average="macro", labels=LABELS, zero_division=0), 1),
         "route_accuracy": round(100*sum(ROUTE[g]==ROUTE[p] for g,p in zip(gold,preds))/len(gold), 1),
         "escalation_recall": round(100*sum(ROUTE[p]=="escalation" for g,p in zip(gold,preds) if ROUTE[g]=="escalation")/sum(ROUTE[g]=="escalation" for g in gold), 1),
         "p50_ms": round(statistics.median(lats)), "p95_ms": round(sorted(lats)[int(.95*len(lats))-1]),
         "usd_per_1k": usd_per_1k if usd_per_1k is not None else round(statistics.mean(lats)/1000/3600*GPU_USD_PER_HOUR*1000, 4),
         "confusion": confusion_matrix(gold, preds, labels=LABELS).tolist()}
    print({k: v for k, v in r.items() if k != "confusion"}); return r
results = []
p, l = predict(model, test); results.append(summarise("Qwen2.5-1.5B base (zero-shot)", p, l))
p, l = predict(model, test, few_shot=True); results.append(summarise("Qwen2.5-1.5B base (10-shot)", p, l))

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth", random_state=2026)
fmt = lambda rows: Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows])
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=fmt(train), eval_dataset=fmt(val),
    args=SFTConfig(output_dir="out", dataset_text_field="text", max_seq_length=512, per_device_train_batch_size=16,
        gradient_accumulation_steps=1, num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
        logging_steps=20, eval_strategy="epoch", save_strategy="no", fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(), seed=2026, report_to=[], packing=False, padding_free=False))
t0 = time.time(); stats = trainer.train(); train_minutes = round((time.time()-t0)/60, 1)
print("train minutes", train_minutes, trainer.state.log_history[-1])

In [ ]:
p, l = predict(model, test); results.append(summarise("Qwen2.5-1.5B + LoRA (fine-tuned)", p, l))
tuned_preds = p

In [ ]:
meta = {"base": BASE, "train_rows": len(train), "val_rows": len(val), "test_rows": len(test), "epochs": 2, "lora_r": 16,
        "train_minutes": train_minutes, "gpu": torch.cuda.get_device_name(0), "gpu_usd_per_hour_assumption": GPU_USD_PER_HOUR,
        "groq_price_per_mtok": [0.15, 0.60]}
json.dump({"meta": meta, "labels": LABELS, "results": results}, open("results.json", "w"), indent=2)
md = "| Model | Accuracy | Macro-F1 | Route acc. | Escalation recall | p50 | p95 | $ / 1k msgs |\n|---|---|---|---|---|---|---|---|\n"
md += "".join(f"| {r['model']} | {r['accuracy']}% | {r['macro_f1']}% | {r['route_accuracy']}% | {r['escalation_recall']}% | {r['p50_ms']} ms | {r['p95_ms']} ms | ${r['usd_per_1k']} |\n" for r in results)
open("RESULTS.md", "w").write(md); print(md)
import matplotlib.pyplot as plt, numpy as np
for r in results:
    cm = np.array(r["confusion"]); fig, ax = plt.subplots(figsize=(7,6)); ax.imshow(cm, cmap="Greens")
    ax.set_xticks(range(len(LABELS)), LABELS, rotation=60, ha="right"); ax.set_yticks(range(len(LABELS)), LABELS)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)): ax.text(j, i, cm[i,j], ha="center", va="center", fontsize=8)
    ax.set_title(r["model"]); fig.tight_layout(); fig.savefig("cm_" + "".join(c if c.isalnum() else "_" for c in r["model"])[:40] + ".png", dpi=120)

In [ ]:
meta["test_sha256"] = TEST_SHA
json.dump({"meta": meta, "labels": LABELS, "results": results}, open("results.json", "w"), indent=2)
model.save_pretrained("keel-router-lora"); tok.save_pretrained("keel-router-lora")
print("KEEL_RESULTS_BEGIN")
print(json.dumps({"meta": meta, "results": [{k: v for k, v in r.items()} for r in results]}))
print("KEEL_RESULTS_END")